# Hyperparameter tuning - approach C (ResNet18, ImageNet fine-tune)

Grid: learning rate {1e-4, 3e-4, 1e-3} x weight decay {0, 1e-4} = 6 runs,
40 epochs each, seed 42, same split and augmentation as the main study.

**Selection is on validation accuracy only.** Test accuracy is recorded for
the report but never used to pick the winner.

Each run writes `results/province_study/tune_lr<lr>_wd<wd>/` and the driver
collects them into `results/province_study/tuning.csv`. ~10 min per run on a T4.


In [ ]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-b00930c9-58d8-8c7e-54ef-c7e7e957cee1)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BUNDLE = '/content/drive/MyDrive/ALPR/alpr_colab_bundle.zip'

import os, zipfile, shutil
shutil.rmtree('/content/alpr', ignore_errors=True)
os.makedirs('/content/alpr', exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    z.extractall('/content/alpr')
%cd /content/alpr

Mounted at /content/drive
/content/alpr


In [ ]:
# ---- BUNDLE FRESHNESS CHECK -------------------------------------------------
import os, glob
assert os.path.exists('scripts/tools/tune_province.py'), 'STALE BUNDLE - rebuild with --province-only'
tr = open('scripts/recognition/train_province_classifier.py', encoding='utf-8').read()
assert '--weight-decay' in tr and '--seed' in tr, 'STALE BUNDLE - rebuild with --province-only'
print('crops:', len(glob.glob('data/province_crops/train/*/*.jpg')), 'train /',
      len(glob.glob('data/province_crops/test/*/*.jpg')), 'test')
print('Bundle is current.')

crops: 2515 train / 567 test
Bundle is current.


In [ ]:
!pip -q install torch torchvision pyyaml tqdm pillow opencv-python-headless numpy

In [ ]:
# ---- THE GRID (skips any run whose run.json already exists) -------------------
!python scripts/tools/tune_province.py --lrs 1e-4 3e-4 1e-3 --wds 0 1e-4 --epochs 40 --seed 42


[tune] tune_lr0.0001_wd0
 TRAIN PROVINCE CLASSIFIER (resnet18, 26 classes)
   device cuda | epochs 40 | batch 32
   output tune_lr0.0001_wd0.pth | mode=fine-tuning | framing_aug=True | rotate=False | rotate180=False | seed=42
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
train=2515 val=529 test=567 | classes=26 | idx_to_class=[0, 1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 2, 20, 21, 22, 23, 24, 25, 3, 4, 5, 6, 7, 8, 9]
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M

In [ ]:
# ---- OPTIONAL: seed variance on the selected config ---------------------------
# Re-run the validation winner with two more seeds so the report can say how much
# of a 0.5-point gap is noise. Fill LR / WD from the table above, then un-comment.
# LR, WD = 1e-4, 0
# for s in (1, 2):
#     !python scripts/recognition/train_province_classifier.py --arch resnet18 --pretrained \
#         --epochs 40 --lr {LR} --weight-decay {WD} --seed {s} \
#         --run-name C_resnet18_finetune_s{s} --out models/recognition/prov_C_s{s}.pth

In [ ]:
# ---- SAVE TO DRIVE --------------------------------------------------------------
import shutil, os, glob
dst = '/content/drive/MyDrive/ALPR/province_study_tuning'
shutil.rmtree(dst, ignore_errors=True)
os.makedirs(dst, exist_ok=True)
for d in glob.glob('results/province_study/tune_*') + glob.glob('results/province_study/C_resnet18_finetune_s*'):
    shutil.copytree(d, os.path.join(dst, os.path.basename(d)),
                    ignore=shutil.ignore_patterns('last.pth'))
shutil.copy('results/province_study/tuning.csv', dst)
print('saved ->', dst)
print('Download this folder and merge it into results/province_study/ in the repo.')

saved -> /content/drive/MyDrive/ALPR/province_study_tuning
Download this folder and merge it into results/province_study/ in the repo.
